# M3L2 E00 - PromptTemplate: del string al componente

## Que vamos a ver

Este notebook muestra el primer paso para pasar de un script tactico a un pipeline orquestado:
reemplazar la concatenacion manual de strings por un `ChatPromptTemplate`.

## Por que empezar por aqui

En la lecture vimos que los scripts RAG tienen un problema tipico:

> "Prompts duplicados, funciones gigantes, llamadas hardcoded al modelo,
> logica de retrieval mezclada con business logic."
> — Lecture Seccion 3.4

El primer sintoma suele ser el prompt: un f-string largo que se repite en varios lugares
y nadie sabe exactamente como funciona.

## Este notebook NO necesita API key

Solo construimos y formateamos prompts. No llamamos al modelo todavia.
Eso viene en E01.


## El problema que viene de M3L1: prompts duplicados (Lecture M3L2 - Seccion 3.4)

En M3L1 construimos prompts del agente asi:

```python
prompt = f"[Thought] {pensamiento}\n[Action] {herramienta}({args})\n[Observation] ..."
```

Eso esta bien para un ejercicio. El problema es cuando crece:

| Sintoma (Lecture Seccion 3.4) | Como aparecia en M3L1 |
|---|---|
| Prompts duplicados | El formato Thought/Action/Obs hardcodeado en varias funciones |
| Funciones gigantes | `agente_manual()` mezclaba prompt + logica + llamada al modelo |
| Llamadas hardcoded | `openai.chat.completions.create(model="gpt-4o-mini", ...)` en cada funcion |
| Dificil debugging | Para ver el prompt habia que agregar prints en cada lugar |
| Poca reutilizacion | El template del agente no se podia separar del resto del codigo |

La lecture dice:

> "El equipo termina invirtiendo demasiado tiempo en entender como funciona el codigo,
> en vez de mejorar el producto."
> -- Lecture M3L2, Seccion 3.5

`ChatPromptTemplate` resuelve el primer punto: convierte el prompt en un objeto
separado con variables explicitas. El resto del pipeline no sabe como esta formado el prompt.


## Mapa de conceptos

| Concepto de la lecture | Pregunta guia | En este notebook |
|---|---|---|
| PromptTemplate | Como estructuro un prompt sin hardcodear texto? | `ChatPromptTemplate.from_messages()` |
| Variables explicitas | Cuales son los inputs del prompt? | `{context}` y `{question}` |
| Reutilizacion | Puedo usar el mismo prompt en varios lugares? | Si, el template es un objeto |
| Modularidad | Puedo cambiar el prompt sin tocar el modelo? | Si, son componentes separados |

Flujo de esta lecture:

```text
Input → PromptTemplate → LLM → OutputParser → Respuesta
        ^^^^^^^^^^^^^^^
        Este notebook se enfoca aqui
```


## Donde encaja PromptTemplate en el pipeline (Lecture M3L2 - Seccion 8.1)

```text
Pipeline simple (M3L2):
  Input -> PromptTemplate -> LLM -> OutputParser -> Respuesta
  ^^^^^^^^^^^^^^^^^^^^
  Esto es lo que practicamos en este notebook

Pipeline RAG (M3L2):
  Input -> Retriever -> Docs -> PromptTemplate -> LLM -> OutputParser -> Respuesta

Pipeline con agente (M3L1 manual / M3L2 E00):
  Input -> Agent -> Tool selection -> Tool execution -> LLM -> Respuesta
                                                        ^^^^
                                              el LLM tambien usa PromptTemplate internamente
```

**Conexion con M3L1**: cuando en M3L1 escribiamos:
```python
system_msg = "Eres un asistente de RRHH..."
prompt_text = system_msg + "\n\n" + "Contexto: " + context + "\n\nPregunta: " + question
```
...eso es exactamente lo que reemplaza `ChatPromptTemplate.from_messages([...])`.


## Bloque 1 - El problema: string concatenado

Asi se suele escribir un prompt en un script tactico.

Ejecuta esta celda y fijate en los problemas que tiene.


In [ ]:
# --- SIN LANGCHAIN: el prompt como string manual ---

context = "La politica de vacaciones es de 15 dias por ano."
question = "Cuantos dias de vacaciones tengo?"

# Version 1: f-string simple
prompt_v1 = f"Contexto: {context}\nPregunta: {question}"
print("Version 1 (f-string simple):")
print(prompt_v1)
print()

# Version 2: concatenacion con \n
system_msg = "Eres un asistente de RRHH. Responde solo con el contexto dado."
prompt_v2 = system_msg + "\n\n" + "Contexto: " + context + "\n\nPregunta: " + question
print("Version 2 (concatenacion):")
print(prompt_v2)
print()

# Problema: si quiero cambiar el formato del prompt, tengo que encontrar
# TODOS los lugares donde use esta logica. No hay una sola fuente de verdad.
print("--- Problemas de este enfoque ---")
print("1. No hay variables explicitas: cualquier string puede ir en context o question")
print("2. El formato del prompt esta mezclado con la logica del programa")
print("3. Si quiero probar el prompt por separado, tengo que extraer el string")
print("4. Si el prompt cambia, busco y reemplazo en todo el codigo")


### Por que esto se vuelve un problema al crecer (Lecture - Seccion 3.3)

Imagina que este prompt se usa en 5 funciones distintas. Ahora quieres agregar:

- instruccion de responder en un idioma especifico,
- formato de salida estructurado,
- manejo del caso donde el contexto esta vacio.

Con strings manuales: buscas y cambias en 5 lugares. Un error tipico es cambiar 4 y olvidar el quinto.

Con `ChatPromptTemplate`: modificas **un objeto** y el cambio aplica en todos lados.


## Bloque 2 - La solucion: ChatPromptTemplate

`ChatPromptTemplate` de LangChain convierte el prompt en un componente con:

- variables **explicitas** (las llaves `{variable}`),
- estructura **declarativa** (system / human / assistant),
- metodo `.format_messages()` para ver el resultado,
- metodo `.invoke()` para usarlo en una chain.

Primero ejecuta la celda de imports.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate


### TODO 1: crear el template

Completa la funcion `build_rag_prompt()` que devuelve un `ChatPromptTemplate`.

El template debe tener:
- un mensaje `system` que explique el rol del asistente,
- un mensaje `human` con las variables `{context}` y `{question}`.

Referencia: `ChatPromptTemplate.from_messages([( "system", "..."), ("human", "...")])`


In [ ]:
def build_rag_prompt() -> ChatPromptTemplate:
    """
    Construye y devuelve un ChatPromptTemplate para responder preguntas con contexto.

    El template debe tener dos variables: {context} y {question}.
    """
    # TODO 1: crear el template usando ChatPromptTemplate.from_messages()
    # Incluir un mensaje system y un mensaje human.
    pass


# Verificacion rapida: si devuelve None, el TODO no esta completo
prompt = build_rag_prompt()
print(f"Tipo del prompt: {type(prompt)}")
print(f"Variables del prompt: {prompt.input_variables if prompt else 'TODO no completado'}")


### TODO 2: formatear el prompt y ver la salida

Usa `.format_messages(context=..., question=...)` para ver como queda el prompt
antes de enviarlo al modelo.

Esto es util para debugging: puedes ver exactamente que texto va a recibir el LLM.


In [ ]:
context_test = "La politica de vacaciones es de 15 dias por ano."
question_test = "Cuantos dias de vacaciones tengo?"

# TODO 2: llamar a prompt.format_messages(context=context_test, question=question_test)
# y guardar el resultado en 'messages'
# messages = ...

# Mostrar cada mensaje formateado
# for msg in messages:
#     print(f"[{msg.type.upper()}] {msg.content}")
#     print()


### TODO 3: probar con distintos contextos

El template funciona como una funcion: el mismo formato, distintos datos.

Ejecuta el template con un contexto diferente y compara la salida.


In [ ]:
# TODO 3: formatear el mismo template con este contexto diferente
context_rrhh = "Los empleados pueden tomar hasta 3 dias de licencia por enfermedad sin certificado."
question_rrhh = "Necesito tomar un dia por enfermedad. Que necesito presentar?"

# messages_rrhh = prompt.format_messages(context=context_rrhh, question=question_rrhh)
# for msg in messages_rrhh:
#     print(f"[{msg.type.upper()}] {msg.content}")
#     print()

print("El mismo template, distintos datos. El formato es siempre consistente.")


## Bloque 3 - Verificar que las variables son explicitas

Una de las ventajas de `ChatPromptTemplate` es que sus variables son declarativas.
Puedes inspeccionarlas, validarlas y documentarlas.


In [ ]:
# Inspeccion del template: esto no es posible con un f-string
if prompt:
    print(f"Variables requeridas: {prompt.input_variables}")
    print(f"Mensajes del template: {len(prompt.messages)}")
    for i, msg in enumerate(prompt.messages):
        print(f"  Mensaje {i}: tipo={msg.__class__.__name__}")
        print(f"            contenido={str(msg.prompt)[:60]}...")
    print()
    print("Con un f-string no puedo saber cuales son las variables sin leer el codigo.")
    print("Con ChatPromptTemplate puedo inspeccionar el objeto programaticamente.")


## Bloque 4 - Checks automaticos


In [ ]:
def run_checks():
    # El template fue creado
    assert prompt is not None, "TODO 1 no completado: prompt es None"
    assert isinstance(prompt, ChatPromptTemplate), "build_rag_prompt debe devolver un ChatPromptTemplate"

    # Las variables correctas estan presentes
    assert "context" in prompt.input_variables, "Falta la variable {context}"
    assert "question" in prompt.input_variables, "Falta la variable {question}"

    # El template tiene al menos dos mensajes (system + human)
    assert len(prompt.messages) >= 2, "El template debe tener al menos un mensaje system y uno human"

    # El formato funciona
    msgs = prompt.format_messages(context="Contexto de prueba.", question="Pregunta de prueba?")
    assert len(msgs) >= 2, "format_messages debe devolver al menos dos mensajes"
    assert "Contexto de prueba." in msgs[-1].content, "El contexto debe aparecer en el mensaje human"
    assert "Pregunta de prueba?" in msgs[-1].content, "La pregunta debe aparecer en el mensaje human"

    print("M3L2 E00 Starter checks passed")


run_checks()


## Cierre - Que aprendimos

| Sin LangChain | Con LangChain (ChatPromptTemplate) |
|---|---|
| Prompt hardcodeado en el flujo. | Prompt como objeto separado. |
| Variables implicitas en el f-string. | Variables explicitas e inspeccionables. |
| Cambio = buscar en todo el codigo. | Cambio = modificar el objeto template. |
| No se puede probar el prompt solo. | Se puede formatear y verificar antes del modelo. |
| Estructura del prompt no es visible. | Estructura declarativa: system / human. |

### Proximo paso: E01

En E01 vamos a conectar este `ChatPromptTemplate` con un `ChatOpenAI`
usando LCEL (`prompt | llm | parser`) para construir la primera chain real.
